# AIC365 Natural Language Processing — Lab 6
## FA23-BAI-006 | Text Classification: Naive Bayes & Logistic Regression

**Scope:** Stage v (Verify) — Home Activities only.

The manual refers to a provided app-review dataset and unseen customer feedback snippets, but those files/snippets were not included in the supplied PDF. This notebook contains a small labelled sample dataset to make the full workflow runnable. Replace it with the official dataset for final course results.

### Setup — sample app-review dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

data = [
("The app is excellent and very easy to use", "positive"),
("Great update, everything works smoothly", "positive"),
("I love the clean design and fast loading", "positive"),
("Very useful app with helpful features", "positive"),
("The support team solved my issue quickly", "positive"),
("Simple to use and reliable every day", "positive"),
("Excellent experience, highly recommended", "positive"),
("The new version is fast and stable", "positive"),
("Good features and a beautiful interface", "positive"),
("Works perfectly and saves me time", "positive"),
("The app crashes every time I open it", "negative"),
("Terrible update, now nothing works", "negative"),
("The app is slow and keeps freezing", "negative"),
("I cannot log in to my account", "negative"),
("Support never replied to my complaint", "negative"),
("This app is full of bugs and errors", "negative"),
("The latest version deleted my saved data", "negative"),
("Very frustrating and difficult to use", "negative"),
("Payment failed and I was charged twice", "negative"),
("The app keeps crashing after the update", "negative"),
("Helpful tool and smooth performance", "positive"),
("Fantastic features and quick support", "positive"),
("Easy setup and good customer service", "positive"),
("The app works well on my phone", "positive"),
("I am happy with the latest update", "positive"),
("Constant errors make this app unusable", "negative"),
("Slow loading and frequent crashes", "negative"),
("I lost my data and cannot restore it", "negative"),
("The payment screen is broken", "negative"),
("Bad experience, the app will not start", "negative"),
("Nice interface and useful reminders", "positive"),
("The service is quick and dependable", "positive"),
("Great app, no problems so far", "positive"),
("It freezes and drains my battery", "negative"),
("Cannot reset password, very annoying", "negative"),
("The new design is clear and attractive", "positive"),
("Customer support was rude and unhelpful", "negative"),
("Works as expected, I recommend it", "positive"),
("This update made the app worse", "negative"),
("Login fails and the screen stays blank", "negative")
]
df = pd.DataFrame(data, columns=["text", "label"])
print("Dataset size:", len(df))
display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))

### Activity 1 — TF-IDF pipeline and evaluation
Train a full pipeline and report accuracy, precision, recall and F1 on a held-out split.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2)),
    ),
    ("clf", LogisticRegression(max_iter=1000))
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision (negative)": precision_score(y_test, y_pred, pos_label="negative", zero_division=0),
    "recall (negative)": recall_score(y_test, y_pred, pos_label="negative", zero_division=0),
    "F1 (negative)": f1_score(y_test, y_pred, pos_label="negative", zero_division=0)
}
display(pd.DataFrame([metrics]).T.rename(columns={0: "score"}))
print(classification_report(y_test, y_pred, zero_division=0))

### Activity 2 — Confusion matrix
Plot the confusion matrix and briefly interpret the errors.

In [ ]:
labels = ["negative", "positive"]
cm = confusion_matrix(y_test, y_pred, labels=labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot()
plt.title("App Review Sentiment — Confusion Matrix")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"Actual negative predicted negative: {tn}")
print(f"Actual negative predicted positive (missed negatives): {fp}")
print(f"Actual positive predicted negative (false alarms): {fn}")
print(f"Actual positive predicted positive: {tp}")
print("Interpretation: inspect the off-diagonal cells to see which sentiment class is being confused. With this small sample, the estimates are illustrative rather than production-level.")

### Activity 3 — Most confident misclassifications
Build the requested DataFrame with text, true label, predicted label and predicted probability.

In [ ]:
probabilities = pipeline.predict_proba(X_test)
classes = list(pipeline.named_steps["clf"].classes_)
predicted_probability = probabilities.max(axis=1)

misclassified = pd.DataFrame({
    "text": X_test.reset_index(drop=True),
    "true_label": y_test.reset_index(drop=True),
    "predicted_label": y_pred,
    "predicted_probability": predicted_probability
})
misclassified = misclassified[misclassified["true_label"] != misclassified["predicted_label"]]
misclassified = misclassified.sort_values("predicted_probability", ascending=False)
display(misclassified.head(5))

### Activity 4 — Class distribution and class-balanced model
Train a second pipeline with `class_weight='balanced'` and compare negative-class precision, recall and F1.

In [ ]:
print("Class distribution:")
display(df["label"].value_counts().rename_axis("label").reset_index(name="count"))

balanced_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
balanced_pipeline.fit(X_train, y_train)
balanced_pred = balanced_pipeline.predict(X_test)

comparison = pd.DataFrame([
    {
        "model": "Original Logistic Regression",
        "negative_precision": precision_score(y_test, y_pred, pos_label="negative", zero_division=0),
        "negative_recall": recall_score(y_test, y_pred, pos_label="negative", zero_division=0),
        "negative_f1": f1_score(y_test, y_pred, pos_label="negative", zero_division=0)
    },
    {
        "model": "Class-weight balanced Logistic Regression",
        "negative_precision": precision_score(y_test, balanced_pred, pos_label="negative", zero_division=0),
        "negative_recall": recall_score(y_test, balanced_pred, pos_label="negative", zero_division=0),
        "negative_f1": f1_score(y_test, balanced_pred, pos_label="negative", zero_division=0)
    }
])
display(comparison)

### Activity 5 — Flag unseen negative feedback
Run the trained original pipeline on sample unseen snippets and provide a short reliability recommendation.

In [ ]:
new_reviews = [
    "The app crashes whenever I try to open my account.",
    "Love the new design, it is simple and fast.",
    "I was charged twice and support has not replied.",
    "Everything works well after the update.",
    "The app is slow and I cannot reset my password."
]
new_predictions = pipeline.predict(new_reviews)
new_probabilities = pipeline.predict_proba(new_reviews)
negative_index = classes.index("negative")

prediction_table = pd.DataFrame({
    "review": new_reviews,
    "predicted_label": new_predictions,
    "negative_probability": new_probabilities[:, negative_index],
    "flag_for_support": [label == "negative" for label in new_predictions]
})
display(prediction_table)

neg_precision = metrics["precision (negative)"]
neg_recall = metrics["recall (negative)"]
print(
    f"Recommendation: Use auto-flagging to prioritize reviews, not to replace human review. "
    f"On this held-out sample, negative-class precision is {neg_precision:.2f} and recall is {neg_recall:.2f}. "
    "Precision indicates how many flagged reviews are truly negative, while recall indicates how many negative reviews are caught. "
    "The dataset is small and illustrative, so evaluate on a larger, representative set before relying on it operationally."
)

### Note
The manual's official app-review dataset and the promised “new unseen customer feedback snippets” were not present in the uploaded PDF. The examples above are clearly marked substitutes. Replace the sample data with the official file/snippets before treating the metrics as coursework results.